Groundwater | Transport Modeling Track

# Step 1: Model Goal – Defining the Transport Model Problem

> **The 10 steps at a glance:** **1-Goal** → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**The journey:** You already know how to define goals for a flow model. Transport modeling introduces new questions – not just *where does the water go?* but *what does it carry along?* This notebook builds on that foundation.

| **Core content** | ~10 minutes |
|:--|:--|
| **Optional: Exercises & real-world context** | +5 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** clear objectives for a transport model **calibrated with heat as a natural tracer**
2. **Identify** what additional data transport modeling requires beyond flow
3. **Explain** why **heat is an ideal calibration tracer** — well-constrained properties and available temperature data
4. **Explain** why a calibrated flow model is the essential foundation for any transport simulation

## Prerequisites

Before starting this notebook, you should have:
- **Completed the Flow Modeling Track** – the transport model builds directly on the calibrated flow model
- Understanding of advection and dispersion concepts from the theory lectures
- Familiarity with the Limmat Valley case study from the flow track

## Introduction

A flow model tells us where water moves and how fast. A transport model asks the next question: **what travels with the water?**

That "what" can be a dissolved substance (a contaminant, a nutrient, a tracer) or **heat itself**. Both are governed by similar mathematics – advection carries them along with the flow, while dispersion and diffusion spread them out – but the parameters and practical implications differ.

In the Limmat Valley, we have river and groundwater temperature data but no solute concentration measurements. **Heat is therefore our natural tracer** — it constrains the transport parameters (effective porosity, dispersivity) that the flow model alone cannot determine. The resulting calibrated model is general: swap the species, define a source term, and it becomes a solute transport model.

In this notebook, we define the goals and scope of our **transport model** for the Limmat Valley aquifer. We already established the flow model's objectives in the flow track; here we focus on what changes when transport enters the picture:
- **What new questions** can we answer?
- **Where** and at what temporal and spatial scale?
- **What additional data** do we need?
- **Who else** cares about the results?

---

## What Do We Need From This Model?

> 💡 **Our Transport Model Requirements**
> 
> We require a **transport model** of the Limmat Valley aquifer to:
> 
> 1. **Calibrate transport parameters** (effective porosity, dispersivity) using heat as a natural tracer — temperature observations from rivers and groundwater monitoring
> 2. **Apply the calibrated model** to scenario analyses: thermal plume propagation and solute transport
> 3. Build on our existing calibrated flow model using **open-source tools** — GWE for heat transport calibration; the same parameter set feeds GWT for solute applications
> 
> The model should be **fast enough for interactive exploration**.

---

## 🎯 Mission Objectives – What Are We Solving?

The flow model answered: *where does the water go and how much?* Transport modeling opens up a different set of questions.

### Primary Objectives — Heat Transport (Calibration Tracer)

Our Limmat Valley heat transport model should be able to:

1. **Simulate thermal plume propagation** from heat injection or extraction
2. **Evaluate thermal impacts** of groundwater heat pump installations
3. **Assess thermal interference** between neighbouring systems
4. **Quantify river–aquifer thermal exchange** and its seasonal dynamics

**Why heat first?** Calibrating with heat constrains the velocity field, effective porosity, and dispersivity — parameters that transfer directly to solute transport. The calibrated model is general-purpose.

<details>
<summary><strong>Application: Solute Transport Scenarios</strong></summary>

With the heat-calibrated transport model, students can address these questions in the case study:

1. **Simulate contaminant plume migration** from hypothetical source locations
2. **Predict arrival times** at receptors (pumping wells, rivers, springs)
3. **Delineate capture zones** and wellhead protection areas
4. **Evaluate remediation scenarios** (e.g., pump-and-treat effectiveness)

</details>

### Educational Objectives

As a teaching tool, the transport model should also:

- Illustrate how the **flow field controls transport**
- Show the roles of **advection, dispersion, and diffusion**
- Demonstrate differences between **conservative and reactive** transport
- Highlight the importance of **numerical stability** (Peclet and Courant criteria)

<details>
<summary><strong>Optional: Exercise – Solute vs. Heat Transport</strong></summary>

> ✏️ **Exercise: Solute vs. Heat Transport**
> 
> Both solute and heat transport are governed by advection-dispersion equations, but they differ in important ways:
> 
> 1. Heat exchanges between the water and the solid matrix much faster than most solutes sorb. What does this imply for how fast a thermal plume moves compared to a solute plume?
> 2. Unlike most contaminants, temperature is naturally present everywhere in the aquifer. How does this affect how we define “background” and “impact”?
> 3. Name one practical application where solute transport matters but heat does not, and one where heat matters but solute does not.
>
> **Hint:** Think about *retardation* – both sorption (solutes) and thermal equilibration (heat) slow down the transport front relative to the groundwater velocity.

</details>

---

## 🗺️ Setting the Scene – Where and When?

We use the same Limmat Valley domain as the flow model, but transport processes place different demands on resolution and time.

### Spatial Scale

| Dimension | Value | Rationale |
|-----------|-------|-----------|
| **Domain extent** | Same as flow model | Full valley, or subset around area of interest |
| **Grid resolution** | 10–50 m near sources | Sharper concentration/temperature gradients require finer cells |
| **Vertical layers** | Potentially more than flow | Capture vertical stratification of plumes |

### Temporal Scale

| Aspect | Choice | Rationale |
|--------|--------|-----------|
| **Simulation type** | Transient | Plumes and thermal fronts evolve over time |
| **Time step** | Days to weeks | Must resolve transport dynamics (Courant criterion) |
| **Simulation period** | Months to decades | Thermal calibration: seasonal to multi-year; solute scenarios: years to decades |

> **Key insight:** Transport simulations are almost always **transient** – even when the underlying flow is steady-state. A contaminant plume at 5 years looks very different from one at 50 years.

---

## Essential Data – What Do We Need?

The flow model already supplies the velocity field. Transport modeling adds its own data requirements.

### Data for Heat Transport (Calibration)

| Category | Data Type | Source | Availability |
|----------|-----------|--------|--------------|
| **Thermal properties** | Thermal conductivity of solids *λ*<sub>s</sub> | Literature | Good |
| | Specific heat capacity of solids *c*<sub>s</sub> | Literature | Good |
| | Density of solids *ρ*<sub>s</sub> | Literature | Good |
| **Porosity** | Effective porosity *n* | Literature, lab tests | ⚠️ Estimated |
| **Dispersivity** | Longitudinal *α*<sub>L</sub>, transverse *α*<sub>T</sub> | Tracer tests, literature | ⚠️ Uncertain |
| **Source terms** | Injection/extraction temperature, flow rate | System design | Known |
| **Background** | Ambient groundwater temperature | Monitoring | Good |
| **Observations** | Temperature measurements | Monitoring wells, river gauges | Good |

<details>
<summary><strong>Additional data for solute transport applications</strong></summary>

| Category | Data Type | Source | Availability |
|----------|-----------|--------|--------------|
| **Diffusion** | Molecular diffusion *D*<sub>m</sub> | Literature | Good |
| **Source terms** | Location, concentration, release history | Site investigations | Case-dependent |
| **Reactions** | Sorption, decay (if applicable) | Lab studies | Optional |
| **Observations** | Concentration measurements | Monitoring wells | Often sparse |

</details>

> **Key terms:**
> - *Effective porosity (n)*: The fraction of pore space through which water (and solutes) actually flows. Controls transport velocity via *v* = *q* / *n*.
> - *Dispersivity (α)*: A length scale that governs mechanical mixing. Notoriously scale-dependent – values measured in lab columns don't apply at field scale.
> - *Thermal conductivity (λ<sub>s</sub>)*: How easily heat moves through the solid–water mixture. Unlike dispersivity, thermal properties of common aquifer materials are well-constrained.

### Data Quality Considerations

- **Heat is well-constrained**: Thermal conductivity and heat capacity of common geological materials fall in narrow, well-studied ranges — a factor of 2–3 covers most cases. This makes heat transport the natural starting point for calibration.
- **Temperature data are available**: River and groundwater temperature observations provide direct calibration targets for the transport model.
- **Scale dependence**: Dispersivity increases with observation scale – but heat calibration constrains it at field scale, avoiding the lab-to-field extrapolation problem.
- **Source characterization**: For solute transport applications, the source term is often the dominant uncertainty — but this is not an issue for heat calibration, where boundary temperatures are measured.
- **Observations**: Concentration data are typically sparser and more expensive than temperature measurements.

### Comprehension Check

Before moving on, make sure you can answer this question:

> **Why are thermal properties of the aquifer matrix generally better constrained than dispersivity, even though both appear in transport equations?**

<details>
<summary>Click to check your answer</summary>

**Thermal conductivity and heat capacity** of common geological materials (sand, gravel, clay) fall in narrow, well-studied ranges – a factor of 2–3 covers most cases. They are intrinsic material properties that don’t depend on the scale of observation.

**Dispersivity**, on the other hand, is *scale-dependent*: it increases with the distance a solute has traveled, because larger transport distances sample more heterogeneity. A lab value of 1 cm may become 10–100 m at field scale. This makes dispersivity one of the most uncertain parameters in solute transport modeling.

This is also why heat transport models are often considered more predictable than solute transport models for the same aquifer.
</details>

---

## 👥 Key Players – Who Cares About the Results?

Transport results often have higher stakes than flow-only results – contamination affects human health, and thermal regulation may have legal requirements.

### Primary Users

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Students** | Learning transport concepts | Clear plume visualizations, interactive scenarios |
| **Instructors** | Teaching advection-dispersion | Illustrative examples for both solute and heat |

<details>
<summary><strong>Optional: Real-world stakeholders</strong></summary>

### In a Real-World Context

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Regulators** | Compliance with concentration/temperature limits | Conservative predictions, uncertainty bounds |
| **Site owners** | Contamination liability, remediation costs | Plume delineation, cleanup time estimates |
| **Water utilities** | Wellhead protection, safe yield | Capture zone analysis, early warning |
| **Energy planners** | Geothermal potential, thermal interference | Temperature plume extent, sustainability |

</details>

---

## Summary: Transport Model Goal Definition

> **🎯 Mission**
> 
> **Calibrate** transport with heat as a natural tracer → **apply** to thermal and solute scenarios
> - Primary: constrain effective porosity and dispersivity using temperature observations
> - Application: thermal plume analysis, and (with added source terms) solute transport
>
> **🗺️ Setting the Scene**
>
> - Spatial: Same domain, finer grid near sources (10–50 m)
> - Temporal: Transient simulations – seasonal to multi-year for calibration, years to decades for scenarios
>
> **📊 Essential Intel**
>
> - Foundation: Calibrated flow model + temperature observations
> - Solute applications add source characterisation and species-specific parameters
>
> **🔑 Key Message**
>
> Heat calibration constrains transport parameters that transfer to any species
>
> **👥 Key Players**
>
> - Primary: Students and instructors (educational use)
> - Real-world: Regulators, site owners, water utilities, energy planners

---

## What You're Taking Forward

The heat-calibrated model gives us a **general transport framework**. For solute applications, we add source terms and species-specific parameters — but the velocity, porosity, and dispersivity are already constrained.

The biggest remaining uncertainty for solute applications is **source characterisation** (what, where, when, how much). Dispersivity — usually the Achilles heel of solute transport — is constrained by heat calibration.

---

## Next Steps

With our transport model goals defined, we're ready to extend our perceptual model to include transport processes – what drives solute and heat movement in the Limmat Valley?

**Continue to:** [2_perceptual_model.ipynb](2_perceptual_model.ipynb)